# Improved CNN Model - Cats vs Dogs

In this notebook, we build an improved CNN model and compare it with the first simple CNN baseline model.

The goal is to reduce overfitting and improve the model's ability to generalize to new images.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
# Project paths

current_path = Path.cwd()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parents[1]

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "dogcat"

TRAIN_DIR = DATA_DIR / "train"
VALIDATION_DIR = DATA_DIR / "validation"

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

print("Project root:", PROJECT_ROOT)
print("Train directory exists:", TRAIN_DIR.exists())
print("Validation directory exists:", VALIDATION_DIR.exists())

Project root: c:\Users\mahta\aidev\cats-dogs-cnn-classifier
Train directory exists: True
Validation directory exists: True


In [5]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_dataset.class_names
print("Class names:", class_names)

Found 25000 files belonging to 2 classes.
Found 8000 files belonging to 2 classes.
Class names: ['cats', 'dogs']


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

## Improvement strategy

The baseline CNN model achieved strong results, but it can still be improved.

The improved model uses three techniques:

1. **Data augmentation**  
   Creates random variations of training images, such as flipping, rotation and zoom.  
   This helps the model become more robust to real-world image variation.

2. **Dropout**  
   Randomly disables some neurons during training.  
   This helps reduce overfitting.

3. **Early stopping**  
   Stops training when the validation loss stops improving.  
   This prevents the model from training too long and memorizing the training data.

In [8]:
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

improved_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

In [9]:
improved_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
improved_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [11]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [12]:
EPOCHS = 3

improved_history = improved_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 189s 240ms/step - accuracy: 0.5663 - loss: 0.6814 - val_accuracy: 0.6261 - val_loss: 0.6715
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 225s 270ms/step - accuracy: 0.6748 - loss: 0.5987 - val_accuracy: 0.6798 - val_loss: 0.6099
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 278s 355ms/step - accuracy: 0.7132 - loss: 0.5537 - val_accuracy: 0.7424 - val_loss: 0.5118


## Local training note

The improved CNN model was first tested locally on CPU for 3 epochs.

Training was slower because TensorFlow GPU support is not available in the local native Windows setup.  
The improved model also includes data augmentation, which makes training heavier.

After 3 epochs, the improved model had lower validation accuracy than the baseline model.  
This is expected because data augmentation makes the task harder at the beginning and usually needs more epochs to show improvement.

For longer training, Google Colab with GPU will be used.